# Embodied Navigation Agent — Colab Training

Run Cell 1 → Cell 2 → Cell 3 (three times).

- Run 1: `STEPS = 5000` (~30s)
- Run 2: `STEPS = 95000` (~8min)
- Run 3: `STEPS = 400000` (~35min)


In [ ]:
# Cell 1 — Install (remove pygame, install headless deps)
# Uninstall pygame completely — we don't need it for training
!pip uninstall -y pygame pygame-ce 2>/dev/null; echo 'pygame removed'
!pip install -q 'stable-baselines3==2.3.2' 'gymnasium==0.29.1' pyyaml
!pip install -q 'minigrid==2.3.1' --no-deps
# Install a stub so minigrid import doesn't fail
!pip install -q pygame-ce --no-deps
print('Done.')

In [ ]:
# Cell 2 — Upload project zip
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
del uploaded

os.makedirs('/content/agent', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/agent')
os.remove(zip_name)

contents = os.listdir('/content/agent')
if len(contents) == 1 and os.path.isdir('/content/agent/' + contents[0]):
    sub = '/content/agent/' + contents[0]
    for item in os.listdir(sub):
        os.rename(sub + '/' + item, '/content/agent/' + item)
    os.rmdir(sub)

print('Ready')

In [ ]:
# Cell 3 — Train
STEPS = 5000  # <-- CHANGE: 5000 / 95000 / 400000

import os
os.makedirs('/content/agent/models', exist_ok=True)
os.makedirs('/content/agent/checkpoints', exist_ok=True)

with open('/content/steps_cfg.py', 'w') as _f:
    _f.write('STEPS = ' + str(STEPS) + '\n')

script = '''
import os, sys

# Headless display
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["DISPLAY"] = ":99"

# CRITICAL: force PyTorch to CPU to avoid torch+CUDA segfault on Colab
os.environ["CUDA_VISIBLE_DEVICES"] = ""

sys.path.insert(0, "/content/agent")
sys.path.insert(0, "/content")
os.chdir("/content/agent")

import numpy as np
import yaml
import gymnasium as gym
from itertools import product
from gymnasium import spaces, ObservationWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList, BaseCallback
from minigrid.core.constants import COLOR_NAMES, OBJECT_TO_IDX, COLOR_TO_IDX
from minigrid.core.grid import Grid
from minigrid.core.mission import MissionSpace
from minigrid.core.world_object import WorldObj
from minigrid.minigrid_env import MiniGridEnv

SEM = {"chair":"ball","table":"key","lamp":"goal","door":"door","box":"box"}
OBJS = frozenset(SEM.keys())

class Obj(WorldObj):
    def __init__(self, s, c):
        object.__init__(self)
        self._n = SEM[s]; self.type = s; self.color = c
        self.contains = self.init_pos = self.cur_pos = None
    def encode(self):
        return (OBJECT_TO_IDX[self._n], COLOR_TO_IDX[self.color], 0)
    def render(self, img): pass

class Env(MiniGridEnv):
    def __init__(self, gs, oc):
        super().__init__(mission_space=MissionSpace(mission_func=lambda: "nav"),
                         grid_size=gs, max_steps=10000, render_mode=None,
                         see_through_walls=False)
        self._oc = oc; self._cc = 0
        self.action_space = spaces.Discrete(4)
    def _gen_grid(self, w, h):
        self.grid = Grid(w, h)
        self.grid.wall_rect(0, 0, w, h)
        combos = list(product(sorted(OBJS), COLOR_NAMES))
        for i in self.np_random.choice(len(combos), size=self._oc, replace=False):
            s, c = combos[i]
            self.place_obj(Obj(s, c), top=(1,1), size=(w-2,h-2))
        self.place_agent(top=(1,1), size=(w-2,h-2), rand_dir=True)
        self.mission = "nav"
    def reset(self, **kw):
        self._cc = 0
        return super().reset(**kw)
    def step(self, a):
        if a == 3:
            return self.gen_obs(), 0.0, True, False, {"collision_count": self._cc}
        pb = tuple(self.agent_pos)
        o, r, t, tr, info = super().step({0:2,1:0,2:1}[a])
        if a == 0 and tuple(self.agent_pos) == pb: self._cc += 1
        info["collision_count"] = self._cc
        return o, r, t, tr, info

GD = 64
gv = np.random.default_rng(99).random(GD).astype(np.float32)
_fl = None; _pr = None

def enc(img):
    global _fl, _pr
    f = img.astype(np.float32).reshape(-1) / 255.0
    if _fl != f.shape[0]:
        _fl = f.shape[0]
        _pr = np.random.default_rng(42).standard_normal((_fl,512)).astype(np.float32)/np.sqrt(512)
    e = f @ _pr; n = np.linalg.norm(e)
    return e/n if n>0 else e

class FE(ObservationWrapper):
    def __init__(self, e):
        super().__init__(e)
        self.observation_space = gym.spaces.Box(-np.inf, np.inf, (512+GD,), dtype=np.float32)
    def observation(self, o):
        img = o["image"] if isinstance(o, dict) else o
        return np.concatenate([enc(img), gv])

with open("configs/default.yaml") as f: cfg = yaml.safe_load(f)
env = FE(Env(cfg["env"]["grid_size"], cfg["env"]["object_count"]))
MODEL = "models/policy.zip"
pc = cfg["ppo"]

if os.path.exists(MODEL):
    policy = PPO.load(MODEL, env=env, device="cpu")
    print("Loaded policy")
else:
    policy = PPO("MlpPolicy", env, device="cpu",
                 learning_rate=pc["learning_rate"], n_steps=pc["n_steps"],
                 batch_size=pc["batch_size"], n_epochs=pc["n_epochs"],
                 gamma=pc["gamma"], gae_lambda=pc["gae_lambda"], clip_range=pc["clip_range"])
    print("Created policy")

class Log(BaseCallback):
    def __init__(self): super().__init__(); self.r=[]; self.l=[]; self.s=[]; self.cr=0.0; self.cl=0
    def _on_step(self):
        self.cr += float(self.locals.get("rewards",[0])[0]); self.cl += 1
        if self.locals.get("dones",[False])[0]:
            self.r.append(self.cr); self.l.append(self.cl); self.s.append(self.cr>=10.0)
            self.cr=0.0; self.cl=0
        return True
    def stats(self):
        lr=self.r[-100:] or [0]; ll=self.l[-100:] or [0]; ls=self.s[-100:] or [False]
        return sum(lr)/len(lr), sum(ll)/len(ll), sum(ls)/len(ls), len(self.r)

import steps_cfg
STEPS = steps_cfg.STEPS
print("Training " + str(STEPS) + " steps on CPU...")
log = Log()
ckpt = CheckpointCallback(save_freq=max(500,STEPS//5), save_path="checkpoints", name_prefix="c", verbose=0)
policy.learn(STEPS, callback=CallbackList([log,ckpt]), reset_num_timesteps=False, progress_bar=False)
policy.save(MODEL)
mr,ml,ms,me = log.stats()
print("Done. reward=" + str(round(mr,3)) + " ep_len=" + str(round(ml,1)) + " success=" + str(round(ms*100,1)) + "% eps=" + str(me))
'''

with open('/content/agent/colab_train.py', 'w') as f:
    f.write(script)

print('Running ' + str(STEPS) + ' steps on CPU...')
!python /content/agent/colab_train.py

In [ ]:
# Cell 4 — Download
from google.colab import files
import os
path = '/content/agent/models/policy.zip'
print('Downloading ' + str(os.path.getsize(path)//1024) + ' KB...')
files.download(path)